# Step 2. Register

*With IMPACT-Reg, from KonfAI: the only step that uses it.*

**Goal:** align round R1 onto round R0, with one command, and get the transformation in OME-Zarr.

With a GPU, replace `--cpu 1` by `--gpu 0`. The first run takes longer: it downloads the preset and the elastix binary.

In [1]:
!impact-reg-konfai register Generic_Rigid \
    -f ../output/R0.ome.zarr -m ../output/R1.ome.zarr -o ../output/registration --cpu 1 --fields-only \
    --set Predictor.outputs_dataset.DisplacementField.OutputDataset.dataset_filename=Transform:omezarr

[ImpactReg] Registration completed in 12 s
  Preset:    Generic_Rigid (Generic Rigid)
  Fixed:     /path/to/ome-zarr-registration-tutorial/output/R0.ome.zarr
  Moving:    /path/to/ome-zarr-registration-tutorial/output/R1.ome.zarr
  Transform: /path/to/ome-zarr-registration-tutorial/output/registration/P000/Transform.ome.zarr
  Record:    /path/to/ome-zarr-registration-tutorial/output/registration/register.json


The command prints its progress, then the summary kept above.

- **`-f` is the fixed image, `-m` the moving one.** The result maps points of the fixed image to the moving image.
- **The images are read from their stores.** The command measures the GPU and the memory it has. A pair too large for them is registered on a coarser grid, then refined on native tiles, with the same command.
- **The preset's lengths are in voxels,** so it needs no tuning for a voxel of 0.2 µm or of 2 mm.
- **`--fields-only` skips the resampled copy** of the moving image: this tutorial does without it, and steps 3 and 4 show how.
- **The last line sets the preset's output dataset.** `Transform:omezarr` writes the transformation in OME-Zarr. Without it, the preset writes `Transform.h5`, as ITK does.

You get two things:

In [2]:
!ls ../output/registration ../output/registration/P000

../output/registration:
P000  register.json

../output/registration/P000:
Transform.ome.zarr


- `Transform.ome.zarr`: the transformation, an RFC-5 `affine`, in micrometers. A rigid registration is six parameters, so the preset declares `transform_type: affine` and writes the matrix it found: twelve numbers, in `zarr.json`, no array. A deformable preset writes a `displacements` field instead, an image of vectors on the fixed grid;
- `register.json`: what ran, with which versions.

## Try another preset

The engine is just a name. `impact-reg-konfai list` shows them all, and `impact-reg-konfai show <preset>` what each one does and what it needs.

| Preset | Engine | Transformation | Time on this pair | Correlation | Nuclei overlap | Runs on |
|---|---|---|---|---|---|---|
| `Generic_Rigid` | elastix, rigid | `affine` | 15 s, CPU | 0.90 | 0.92 | any machine |
| `Generic_Rigid_BSpline` | elastix, rigid then B-spline | `displacements` | 2 min, CPU | 0.82 | 0.73 | any machine |
| `ConvexAdam_Composite` | ConvexAdam on MIND features | `displacements` | 1 min, GPU | 0.99 | 0.94 | Linux and Windows, no macOS package |
| `FireANTs_SyN` | FireANTs, diffeomorphic | `displacements` | 2 min 30, GPU | 0.99 | 0.94 | a GPU: over 50 min on a CPU |

Deformable is not always better: here, the B-spline scores below the rigid it starts from.

The scores come from step 4. Correlation starts at -0.06, and the overlap at 0. With a `displacements` field, step 3 copies it below the scene, as `steps/write.py` does; the notebook shows the affine.

These presets declare `moments_init: com`: the moving image starts where the two subjects' centres of mass coincide. For two images covering different lengths of a body, add `--set moments_init=cof`, which aligns the two frames instead. On this pair, that leaves the organoid where it starts, at -0.06.

**Next: [Step 3. Write, and read back](3-write.ipynb)**